In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
cfg = common.load_config(dbutils.widgets.get("env"))
g, s = f"{cfg['catalog']}.gold", f"{cfg['catalog']}.silver"
results = []

def check(name, ok, detail=""):
    results.append(bool(ok))
    print("PASS" if ok else "FAIL", name, detail)

def one(sql):
    return spark.sql(sql).first()[0]

In [0]:
from pyspark.sql import functions as F
from lh import common
cat = common.load_config(dbutils.widgets.get("env"))["catalog"]
g, f = f"{cat}.gold", f"{cat}.gold.fact_bhg_event"
one = lambda sql: spark.sql(sql).first()[0]
results = []

def check(name, ok, detail=""):
    results.append(bool(ok))
    print(("PASS " if ok else "FAIL ") + name + (f" {detail}" if detail != "" else ""))

n, keys = one(f"SELECT count(*) FROM {f}"), one(f"SELECT count(DISTINCT event_key) FROM {f}")
check("9306 events, one key each", n == 9306 and keys == n, (n, keys))
by_type = {r[0]: r[1] for r in spark.sql(f"SELECT event_type, count(*) FROM {f} GROUP BY 1").collect()}
check("events per type", by_type == {"consult": 1900, "treatment": 506, "room": 268, "pharmacy": 1644,
                                     "discount": 845, "tax": 2168, "payment": 1975}, by_type)

# every invoice: charges + adjustments = net; paid invoices: payments = net,
# except a paid invoice whose payment is held in quarantine (rule future_date) - that gap is expected
breaks = spark.sql(f"""
    WITH c AS (SELECT invoice_id,
                      SUM(CASE WHEN event_group IN ('charge', 'adjustment') THEN amount END) AS billed,
                      COALESCE(SUM(CASE WHEN event_group = 'collection' THEN amount END), 0) AS paid
               FROM {f} WHERE invoice_id IS NOT NULL GROUP BY invoice_id)
    SELECT i.invoice_id, ABS(c.billed - i.net_amt) > 0.01 AS bill_break,
           i.status = 'paid' AND ABS(c.paid - i.net_amt) > 0.01 AS pay_break
    FROM c JOIN {cat}.silver.invoices i USING (invoice_id)""").filter("bill_break OR pay_break")
held = spark.sql(f"""SELECT raw_row FROM {cat}.silver.quarantine
                     WHERE table_name = 'payments' AND status = 'open' AND rule <> 'orphan'""")
explained = breaks.filter("NOT bill_break").join(held, F.expr("raw_row LIKE concat('%', invoice_id, '%')"), "left_semi")
unexplained = breaks.join(explained, "invoice_id", "left_anti").count()
check("every invoice reconciles (a payment held in quarantine explains its gap)", unexplained == 0,
      f"breaks {breaks.count()}, explained by held payments {explained.count()}")

for key, dim in [("date_key", "dim_date"), ("patient_key", "dim_patient"), ("doctor_key", "dim_doctor"),
                 ("hospital_key", "dim_hospital"), ("payer_key", "dim_payer")]:
    orphans = one(f"SELECT count(*) FROM {f} e LEFT ANTI JOIN {g}.{dim} d USING ({key}) WHERE e.{key} IS NOT NULL")
    check(f"every {key} exists in {dim}", orphans == 0, orphans)

check("no event without a date or payer", one(f"SELECT count(*) FROM {f} WHERE date_key IS NULL OR payer_key IS NULL") == 0)
unknown = one(f"SELECT count(*) FROM {f} WHERE doctor_key = -1")
check("events of accepted orphans point to the unknown doctor (5)", unknown == 5, unknown)
future = one(f"SELECT count(*) FROM {f} WHERE date_key > CAST(date_format(current_date(), 'yyyyMMdd') AS INT)")
check("no event dated in the future", future == 0, future)

print(f"\nt19_gold_fact_bhg: {sum(results)} of {len(results)} checks passed ->", "PASS" if all(results) else "FAIL")
if not all(results):
    raise RuntimeError("t19_gold_fact_bhg failed")